In [ ]:
# 0. Import libraries
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from optuna.distributions import CategoricalDistribution, FloatDistribution, IntDistribution
import sys
import os
sys.path.append('../src')
from functions import RepeatedNestedCV, summarize_with_ci, train_and_save_final_model

In [ ]:
# 1. Load and preprocess data
data = pd.read_csv('../data/breast_cancer_cleaned.csv')

# 1.1 Split into X and y
X = data.drop(columns=['diagnosis']).values
y = data['diagnosis'].values

In [ ]:
# 2.  Define estimators with preprocessing pipelines

# Models with standardization (StandardScaler):
# - LogisticRegression
# - LDA
# - SVM

# Models without standardization:
# - GaussianNB
# - RandomForest
# - LightGBM

estimators = {
    'LogisticRegression': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(penalty='elasticnet', solver='saga', max_iter=10000))
    ]),
    'GaussianNB': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('clf', GaussianNB())
    ]),
    'LDA': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('clf', LinearDiscriminantAnalysis())
    ]),
    'SVM': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('clf', SVC(probability=True))
    ]),
    'RandomForest': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('clf', RandomForestClassifier())
    ]),
    'LightGBM': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('clf', LGBMClassifier())
    ])
}

In [ ]:
# 3. Define hyperparameter search spaces for each estimator
param_grids = {
    'LogisticRegression': {
        'clf__C': FloatDistribution(1e-2, 10, log=True),
        'clf__l1_ratio': FloatDistribution(0.0, 1.0),
    },
    'GaussianNB': {
        'clf__var_smoothing': FloatDistribution(1e-9, 1e-7, log=True),
    },
    'LDA': {
        'clf__solver': CategoricalDistribution(['svd', 'lsqr', 'eigen']),
        'clf__shrinkage': CategoricalDistribution(['auto', None]),
    },
    'SVM': {
        'clf__C': FloatDistribution(1e-2, 10, log=True),
        'clf__kernel': CategoricalDistribution(['linear', 'rbf']),
    },
    'RandomForest': {
        'clf__n_estimators': IntDistribution(100, 300),
        'clf__max_depth': IntDistribution(3, 15),
        'clf__min_samples_split': IntDistribution(2, 10),
    },
    'LightGBM': {
        'clf__n_estimators': IntDistribution(100, 300),
        'clf__max_depth': IntDistribution(3, 15),
        'clf__learning_rate': FloatDistribution(0.01, 0.2),
    }
}

In [ ]:
# 4. Run nested cross-validation with hyperparameter tuning
nested_cv = RepeatedNestedCV(
    estimators=estimators,
    param_grids=param_grids,
    R=10,
    N=5,
    K=3,
    scoring='balanced_accuracy'
)

results_df = nested_cv.run(X, y)


In [ ]:
# 5. Save all results with their metrics for later use
results_df['Method'] = 'Baseline'
results_df.to_csv("../results/nested_cv_results.csv", index=False)


In [ ]:
# 6. Analyze and visualize metrics
summarize_with_ci(results_df)

In [ ]:
# 7. Define discrete hyperparameter grids for final training with GridSearchCV
param_grids_final = {
    'LogisticRegression': {
        'clf__C': [0.01, 0.1, 1, 10],
        'clf__l1_ratio': [0.0, 0.5, 1.0],
    },
    'GaussianNB': {
        'clf__var_smoothing': [1e-9, 1e-8, 1e-7],
    },
    'LDA': {
        'clf__solver': ['svd', 'lsqr', 'eigen'],
        'clf__shrinkage': ['auto', None],
    },
    'SVM': {
        'clf__C': [0.01, 0.1, 1, 10],
        'clf__kernel': ['linear', 'rbf'],
    },
    'RandomForest': {
        'clf__n_estimators': [100, 200],
        'clf__max_depth': [None, 5, 10],
        'clf__min_samples_split': [2, 5],
    },
    'LightGBM': {
        'clf__n_estimators': [100, 200],
        'clf__max_depth': [-1, 5, 10],
        'clf__learning_rate': [0.01, 0.1],
    }
}


In [ ]:
# 8. Train and save final model selected from best-performing estimator

winner = 'SVM'  
final_model, final_params = train_and_save_final_model(
    X, y,
    estimators=estimators,
    param_grids=param_grids_final,
    winner=winner
)
